# Sampling with `QSamplerV2`

`QSamplerV2` is the Qiskit sampler primitive (`SamplerV2`) for QaaS backends. It returns a `PrimitiveResult`:

- one result per PUB (a circuit, optionally with parameter values and shots),
- one `BitArray` per classical register, named after the register,
- results shaped like the parameter values, for parameter sweeps.

For a plain `qiskit.result.Result` (counts, memory), use `backend.run(...)` and `job.result()` instead, as in `randomnumbers_Qprovider_qheappe.ipynb`.

In [ ]:
%pip install pylatexenc
%pip install matplotlib
%pip install 'qaas[iqm_qiskit]>=1.1.0' --extra-index-url https://opencode.it4i.eu/api/v4/projects/107/packages/pypi/simple

In [ ]:
import os

os.environ["QPROVIDER_LOGLEVEL"] = "INFO"

# Connect to system

In [ ]:
from qheappe_client import QBackend, QProvider, QSamplerV2

lexis_project = "EU-xx-xxx"
resource_name = "VLQ-EU"

provider = QProvider(lexis_project)

## or define manually LEXIS access token e.g. via dotenv

## %pip install python-dotenv
## from dotenv import load_dotenv
## load_dotenv()

## provider = QProvider(LEXIS_PROJECT, token)

In [ ]:
backend: QBackend = provider.get_backend(resource_name, backend_name="VLQ")
sampler = QSamplerV2(backend)  # default shots: backend.options.shots (1024)

print(f"Qubits: {len(backend.architecture.qubits)}")

# 1. Bell state

Circuits must be transpiled for the backend before they are passed to the sampler.

In [ ]:
from qiskit import QuantumCircuit

bell = QuantumCircuit(2, name="bell")
bell.h(0)
bell.cx(0, 1)
bell.measure_all()  # adds the classical register "meas"

bell_isa = backend.transpile(bell)

`wait_for_final_state()` follows HEAppE's event stream, so it returns as soon as the job finishes. It's more efficient than calling `job.status()` in a loop. `job.result()` waits the same way, so calling `wait_for_final_state()` first is optional.

In [ ]:
job = sampler.run([bell_isa], shots=1000)
print("HEAppE job ID(s):", job.job_id())

job.wait_for_final_state(
    timeout=600, callback=lambda job_id, status, job: print(job_id, status.name)
)
print("Final status:", job.status().name)

In [ ]:
result = job.result()
meas = result[0].data.meas  # BitArray of the "meas" register

print("Counts:", meas.get_counts())
print("First 10 shots:", meas.get_bitstrings()[:10])
print("Shots:", meas.num_shots, "| bits:", meas.num_bits)

In [ ]:
from qiskit.visualization import plot_histogram

plot_histogram(meas.get_counts(), figsize=(8, 4), title="Bell state")

# 2. Several classical registers

Each classical register gets its own `BitArray`. Bit 0 is the rightmost bit, as in Qiskit.

In [ ]:
from qiskit import ClassicalRegister, QuantumRegister

a, b = ClassicalRegister(2, "a"), ClassicalRegister(1, "b")
regs = QuantumCircuit(QuantumRegister(3), a, b, name="registers")
regs.x(0)  # expected: a = "01", b = "0"
regs.measure([0, 1], a)
regs.measure(2, b)

data = sampler.run([backend.transpile(regs)], shots=500).result()[0].data
print("Register a:", data.a.get_counts())
print("Register b:", data.b.get_counts())

# 3. Parameter sweep

One PUB with a parameterized circuit and an array of parameter values runs every value in a single HEAppE job. The result has the same shape as the parameter values.

For `rx(θ)` on `|0⟩`, the probability of measuring `1` is `sin²(θ/2)`.

In [ ]:
import numpy as np
from qiskit.circuit import Parameter

theta = Parameter("θ")
rx = QuantumCircuit(1, name="rx_sweep")
rx.rx(theta, 0)
rx.measure_all()

thetas = np.linspace(0, 2 * np.pi, 13)
sweep = sampler.run([(backend.transpile(rx), thetas.reshape(-1, 1))], shots=500).result()[0]

meas = sweep.data.meas
print("Result shape:", meas.shape)  # (13,): one BitArray per angle

p1 = [meas[i].get_counts().get("1", 0) / meas[i].num_shots for i in range(len(thetas))]

In [ ]:
import matplotlib.pyplot as plt

fine = np.linspace(0, 2 * np.pi, 200)
plt.figure(figsize=(8, 4))
plt.plot(fine, np.sin(fine / 2) ** 2, label="expected sin²(θ/2)")
plt.plot(thetas, p1, "o", label="measured")
plt.xlabel("θ (rad)")
plt.ylabel("P(1)")
plt.legend()
plt.show()

# 4. PUBs with different shot counts

Every PUB can set its own shots: `(circuit, parameter_values, shots)`. PUBs with the same shot count share one HEAppE job; different shot counts run as separate jobs.

In [ ]:
job = sampler.run([(bell_isa, None, 200), (bell_isa, None, 800)])
print("HEAppE jobs:", [j.job_id() for j in job.jobs])

for i, pub_result in enumerate(job.result()):
    print(f"PUB {i}: {pub_result.data.meas.num_shots} shots, {pub_result.data.meas.get_counts()}")

# 5. Qiskit's `BackendSamplerV2`

QaaS backends follow Qiskit's `BackendV2` interface, so Qiskit's generic `BackendSamplerV2` also works. It returns the same kind of result as `QSamplerV2`.

In [ ]:
from qiskit.primitives import BackendSamplerV2

generic = BackendSamplerV2(backend=backend).run([bell_isa], shots=1000).result()
print("BackendSamplerV2:", generic[0].data.meas.get_counts())